In [1]:
import attribute_disentanglement.diffusion as K
from attribute_disentanglement.datasets import named_data
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset

2026-03-19 00:17:12.372135: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-19 00:17:12.413390: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-19 00:17:16.878264: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
import lpips
device = "cuda:0"
extractor = lpips.LPIPS(net='vgg').to(device).to(torch.float32)

Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]


/vol/biomedic2/rrr2417/Diffusion-Representation-Learning/.venv/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/vol/biomedic2/rrr2417/Diffusion-Representation-Learning/.venv/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: /vol/biomedic2/rrr2417/Diffusion-Representation-Learning/.venv/lib/python3.10/site-packages/lpips/weights/v0.1/vgg.pth


In [9]:
config_files = {
    "cars3d": ("configs_final/cars3d/A.json_", "ckpts/vqvae/0_cars3d_True_32_128_5e-05_new"),
    "mpi3d": ("configs_final/mpi3d/A.json_", "ckpts/vqvae/0_mpi3d_True_32_128_5e-05_new"),
    "shapes3d": ("configs_final/shapes3d/A.json_", "ckpts/vqvae/0_shapes3d_True_32_128_5e-05_new"),
                 # "/vol/biomedic2/rrr2417/Diffusion-Representation-Learning/experiments/EncDiff/logs2/2026-03-10T22-53-02_shapes3d_vq_4_160/checkpoints/last.ckpt"),
                 # "/vol/biomedic3/rrr2417/encdiff/k-diffusion/ckpts/vqvae/2_shapes3d_True_32_128_5e-05/training-state-020019.pkl"), #"ckpts/vqvae/2_shapes3d_True_32_128_5e-05"),
    "coco": ("configs_final/coco/test0_s6.json_", "ckpts/vqvae/coco"),
    "voc": ("configs_final/voc/test0.json", "ckpts/vqvae/voc"),
    "clevrtex": ("configs_logsnr2/clevrtex/test0.json", "ckpts/vqvae/clevrtex"),
}
ds_name = "mpi3d"
batch_size = 64

In [10]:
# from attribute_disentanglement.vqvae.VQVAE import VQVAE, VQVAEWrapper
# import pickle

# # path = "/vol/biomedic3/rrr2417/encdiff/k-diffusion/ckpts/vqvae/0_shapes3d_True_32_128_5e-05/training-state-090587.pkl"
# # path = "/vol/biomedic3/rrr2417/encdiff/k-diffusion/ckpts/vqvae/0_mpi3d_True_32_128_5e-05/training-state-086583.pkl"
# # path = "/vol/biomedic3/rrr2417/encdiff/k-diffusion/ckpts/vqvae/0_shapes3d_True_32_128_5e-05/training-state-097093.pkl"
# # path = "/vol/biomedic3/rrr2417/encdiff/k-diffusion/ckpts/vqvae/0_shapes3d_True_32_128_5e-05/training-state-115110.pkl"
# # path = "/vol/biomedic3/rrr2417/encdiff/k-diffusion/ckpts/vqvae/0_shapes3d_True_32_128_5e-05/training-state-146640.pkl"
# path = "/vol/biomedic3/rrr2417/encdiff/k-diffusion/ckpts/vqvae/0_shapes3d_True_32_128_5e-05/training-state-150000.pkl"
# # path = "/vol/biomedic3/rrr2417/encdiff/k-diffusion/ckpts/vqvae/2_shapes3d_True_32_128_5e-05/training-state-142136.pkl"
# if ds_name in ['shapes3d', 'mpi3d', 'cars3d']:
#     with open(path, "rb") as f:
#         data = pickle.load(f)
#     vqvae = data["net"].eval()
#     for p in vqvae.parameters():
#         p.requires_grad_(False)

# vqvae = VQVAEWrapper(vqvae).to(device)

In [11]:
config = K.config.load_config(config_files[ds_name][0])
vqvae = K.config.make_vqvae(config).to(device)
dataset_config = config["dataset"]

if dataset_config['type'] in ["shapes3d", "cars3d", "mpi3d"]:
    dataset_obj = named_data.get_named_ground_truth_data(dataset_config['type'])
    _, reals = dataset_obj.sample(30000, np.random.RandomState(0))
    reals = K.utils.normalise(torch.tensor(reals).to(device).to(torch.float32).permute(0, 3, 1, 2))
    reals = TensorDataset(reals)
    dl_train = DataLoader(
        reals,
        batch_size=batch_size,
        num_workers=0,
        drop_last=True,
        shuffle=False,
    )
else:
    from attribute_disentanglement.datasets.ocl import build_dataset
    data_root = {
        'clevrtex': './datasets/CLEVRTex',
        'voc': './datasets/VOC',
        'coco': './datasets/COCO',
    }
    params = dict(
        dataset=dataset_config['type'],
        data_root=data_root[config['dataset']['type']],
        resolution=(128, 128) if config['dataset']['type'] == 'clevrtex' else (224, 224),
        load_mask=True,
        load_anno=True,
    )
    datasets = build_dataset(params)
    train_set, val_set = datasets[0], datasets[1]
    val_set.load_sem_mask = True
    collate_fn = datasets[2] if len(datasets) == 3 else None
    dl_train = DataLoader(
        train_set,
        batch_size=batch_size,
        collate_fn=collate_fn,
        num_workers=0,
        shuffle=False,
        drop_last=True,
    )

Scale Factor: 1.0
Using scale factor 1.0


In [12]:
from tqdm import tqdm

encs = []
n = 0
lpips = 0
for real in tqdm(dl_train):
    if dataset_config['type'] in ["shapes3d", "cars3d", "mpi3d"]:
        real = real[0]
    else:
        real = real["img"].to(device)
    with torch.no_grad():
        enc = vqvae.encode(real) # .cpu()
        recon = vqvae.decode(enc).clamp(-1, 1)
        lpips_ = K.evaluation.perceptual_dist(
            real,
            recon.to(torch.float32),
            extractor,
        )
    # print(real.min(), real.max(), real.mean())
    # print(recon.min(), recon.max(), recon.mean())
    lpips += lpips_
    encs.append(enc.cpu())
    n += real.shape[0]
    if n >= 10000 and dataset_config['type'] not in ["shapes3d", "cars3d", "mpi3d"]:
        break

mean = torch.cat(encs, dim=0).flatten().mean()
std = torch.cat(encs, dim=0).flatten().std()

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 468/468 [00:09<00:00, 48.80it/s]


In [13]:
print(std), print(mean), print(lpips / len(dl_train))

tensor(0.4551)
tensor(0.1536)
tensor(0.0122, device='cuda:0')


(None, None, None)

In [8]:
import os

torch.save(mean, os.path.join(config_files[ds_name][1], "mean.pt"))
torch.save(std, os.path.join(config_files[ds_name][1], "std.pt"))

In [36]:
torch.load(os.path.join(config_files["shapes3d"][1], "mean.pt")), torch.load(os.path.join(config_files["shapes3d"][1], "std.pt"))

(tensor(-0.1324), tensor(0.4086))